In [2]:
import sys
sys.path.append('..')
from groq import Groq
import os
from helpers import *
from dotenv import load_dotenv
import json

In [3]:
load_dotenv()

True

In [4]:
client = Groq(api_key=os.getenv("Groq_API_KEY"))

response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {"role": "user", "content": "Say hello in 5 words"}
    ]
)

print(response.choices[0].message.content)

Hello there, wonderful world today!


In [5]:


def get_user_intent(user_text):
    prompt = f"""
    User said: "{user_text}"
    
    Extract the following and return ONLY valid JSON:
    
    {{
    "mood": one of [Sad, Tired, Neutral, Happy, Excited, Angry, null],
    "genre_preference": genre name or null,
    "reference_title": movie/show name or null,
    "similarity_intent": "similar" or "different" or null,
    "avoid_genres": list of genres to avoid or empty list,
    "vibe": one of [feel-good, suspense, dark, light, emotional, joyful, heartfelt, revenge, inspirational, null],
    "language": one of [English, Hindi, Kannada, Telugu, Tamil, Malayalam, Korean, null]
    }}
    
    Available genres: Action, Adventure, Animation, Children, 
    Comedy, Crime, Documentary, Drama, Fantasy, Film-Noir, 
    Horror, Musical, Mystery, Romance, Sci-Fi, Thriller, War, Western
    
    Note: "rom-com" means Romance + Comedy.
    Only include what's explicitly or implicitly mentioned.

    Note: Only set "language" if the user explicitly mentions a language,
    region, or a reference title strongly associated with one (e.g.
    "Korean movie", "Tamil film", "something like Parasite").
    Otherwise leave it null.
    """
    
    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": prompt}]
    )
    
    raw = response.choices[0].message.content
    
    try:
        return json.loads(raw)
    except Exception as e:
        print(f"Parse error: {e}")  # shows WHY it failed
        return {
            "mood": None, "genre_preference": None,
            "reference_title": None, "similarity_intent": None,
            "avoid_genres": [],
            "vibe" : None,
            "language" : None
        }

In [6]:
test_inputs = [
    "had a rough day",
    "I liked Vincenzo, suggest similar ones",
    "watched too many rom-coms, want something different"
]

for text in test_inputs:
    print(f"INPUT: {text}")
    print(f"OUTPUT: {get_user_intent(text)}")
    print("---")

INPUT: had a rough day
OUTPUT: {'mood': 'Sad', 'genre_preference': None, 'reference_title': None, 'similarity_intent': None, 'avoid_genres': [], 'vibe': None, 'language': None}
---
INPUT: I liked Vincenzo, suggest similar ones
OUTPUT: {'mood': None, 'genre_preference': None, 'reference_title': 'Vincenzo', 'similarity_intent': 'similar', 'avoid_genres': [], 'vibe': None, 'language': 'Korean'}
---
INPUT: watched too many rom-coms, want something different
OUTPUT: {'mood': 'Tired', 'genre_preference': None, 'reference_title': None, 'similarity_intent': 'different', 'avoid_genres': ['Romance', 'Comedy'], 'vibe': None, 'language': None}
---


In [7]:
result = get_user_intent("had a rough day")
print(result)
print("Mood:", result["mood"])

{'mood': 'Sad', 'genre_preference': None, 'reference_title': None, 'similarity_intent': None, 'avoid_genres': [], 'vibe': 'emotional', 'language': None}
Mood: Sad


In [8]:
# For LLM logic (no emojis)
mood_to_genres = {
    "Angry"   : ["Comedy", "Animation", "Musical"],
    "Sad"     : ["Comedy", "Animation", "Musical"],
    "Tired"   : ["Animation", "Comedy", "Children"],
    "Neutral" : ["Action", "Adventure", "Documentary"],
    "Happy"   : ["Comedy", "Romance", "Musical"],
    "Excited" : ["Thriller", "Crime", "Action", "Mystery"]
}

In [9]:
vibe_to_genres = {
    "joyful"        : ["Comedy", "Children", "Musical", "Animation"],
    "suspense"      : ["Thriller", "Mystery", "Crime", "Horror", "Sci-Fi"],
    "revenge"       : ["Action", "Crime", "Thriller", "Drama", "Western"],
    "heartfelt"     : ["Drama", "Romance", "Children", "Adventure"],
    "inspirational" : ["Drama", "Documentary", "Adventure", "War"],
    "feel-good"     : ["Comedy", "Romance", "Children", "Musical", "Adventure"],
    "dark"          : ["Drama", "Thriller", "Crime", "Film-Noir", "Horror", "War", "Sci-Fi"],
    "light"         : ["Comedy", "Children", "Animation", "Musical", "Adventure"],
    "emotional"     : ["Drama", "Romance", "Documentary", "Children"]
}

In [10]:
def resolve_genre(intent, avoided_genres=None):
    if avoided_genres is None:
        avoided_genres = set()

    # 1. Explicit genre — ALWAYS respected, avoid-list ignored 🎯
    if intent.get('genre_preference'):
        return intent['genre_preference']

    # 2. Vibe — pick first NOT avoided
    if intent.get('vibe') and intent['vibe'] in vibe_to_genres:
        for g in vibe_to_genres[intent['vibe']]:
            if g not in avoided_genres:
                return g                      # first survivor wins

    # 3. Mood — same pattern
    if intent.get('mood') and intent['mood'] in mood_to_genres:
        for g in mood_to_genres[intent['mood']]:
            if g not in avoided_genres:
                return g

    # 4. Default
    return "Drama"

In [11]:
# Test: sad mood BUT wants suspense
intent = {
    "genre_preference": None,
    "vibe": "suspense",
    "mood": "Sad"
}
print(resolve_genre(intent))

Thriller


In [12]:
# Test 1 — Sad mood BUT wants suspense
intent = {
    "genre_preference": None,
    "vibe": "suspense",
    "mood": "Sad"
}
print("Sad + suspense →", resolve_genre(intent))
# Should be Thriller (vibe wins!)

Sad + suspense → Thriller


In [13]:
# Test with different intents
test1 = {"genre_preference": "Horror", "mood": None}
print(resolve_genre(test1))  # Horror

test2 = {"genre_preference": None, "mood": "Sad"}
print(resolve_genre(test2))  # Comedy

test3 = {"genre_preference": None, "mood": None}
print(resolve_genre(test3))  # Drama

Horror
Comedy
Drama


In [14]:
def filter_avoid_genres(titles, avoid_genres):
    """
    Remove movies whose genres match 
    any in avoid_genres list.
    """
    if not avoid_genres:
        return titles  # nothing to avoid
    
    filtered = []
    for title in titles:
        # Get this movie's genres
        movie_row = df_movies[df_movies['title'] == title]
        if len(movie_row) == 0:
            continue
        
        movie_genres = movie_row['genres'].values[0]
        
        # Check if ANY avoid genre is in this movie
        should_avoid = False
        for avoid in avoid_genres:
            if avoid in movie_genres:
                should_avoid = True
                break
        
        # Keep only if not avoided
        if not should_avoid:
            filtered.append(title)
    
    return filtered

In [15]:
def get_smart_recommendations(user_text, user_id=None, n=5):
    '''
    Full LLM pipeline: text → intent → genre → recommendations
    '''
    # Step 1 — Extract intent from text
    intent = get_user_intent(user_text)
   

    # Step 2 — Resolve genre
    genre = resolve_genre(intent)
    

    # Get MORE than needed (buffer for filtering)
    buffer_n = n * 4   # e.g. 20 if n=5
    recommendations = get_recommendations(genre=genre, n=buffer_n, user_id=user_id)
    
    # Filter out avoided genres
    avoid = intent.get('avoid_genres', [])
    recommendations = filter_avoid_genres(recommendations, avoid)
    
    # Return top n from filtered
    return recommendations[:n], genre

In [16]:
# Test each piece separately
text = "had a rough day"

# 1. Intent
intent = get_user_intent(text)
print("1. INTENT:", intent)
print("   Type:", type(intent))

# 2. Genre
genre = resolve_genre(intent)
print("2. GENRE:", genre)
print("   Type:", type(genre))

# 3. Recommendations
recs = get_recommendations(genre=genre, n=5, user_id=None)
print("3. RECS:", recs)
print("   Type:", type(recs))

1. INTENT: {'mood': 'Sad', 'genre_preference': None, 'reference_title': None, 'similarity_intent': None, 'avoid_genres': [], 'vibe': None, 'language': None}
   Type: <class 'dict'>
2. GENRE: Comedy
   Type: <class 'str'>
3. RECS: ['Big Chill, The (1983)', 'Evan Almighty (2007)', 'Man, The (2005)', 'Burn After Reading (2008)', 'Man in the White Suit, The (1951)']
   Type: <class 'list'>


In [17]:
recs = get_smart_recommendations("had a rough day")
print(recs)

(['Goodfellas (1990)', 'Prairie Home Companion, A (2006)', 'Out of Sight (1998)', 'Life Is Beautiful (La Vita è bella) (1997)', 'Larry Crowne (2011)'], 'Drama')


In [18]:
test_cases = [
    "had a rough day",
    "I want horror movies",
    "feeling excited and pumped up!",
]

for text in test_cases:
    print(f"\nINPUT: '{text}'")
    recs = get_smart_recommendations(text)
    for r in recs:
        print(f"  → {r}")


INPUT: 'had a rough day'
  → ['5 Centimeters per Second (Byôsoku 5 senchimêtoru) (2007)', 'Miracle on 34th Street (1947)', 'Barfly (1987)', 'Single Man, A (2009)', 'Oversimplification of Her Beauty, An (2012)']
  → Drama

INPUT: 'I want horror movies'
  → ['Dead Fury (2008)', 'Army of Darkness (1993)', 'Friday the 13th (2009)', "Devil's Backbone, The (Espinazo del diablo, El) (2001)", 'Watcher in the Woods, The (1980)']
  → Horror

INPUT: 'feeling excited and pumped up!'
  → ['To Have and Have Not (1944)', 'Mad Max: Fury Road (2015)', 'Boondock Saints II: All Saints Day, The (2009)', 'Friday the 13th (1980)', 'Angels with Dirty Faces (1938)']
  → Thriller


In [19]:
# Test avoid_genres
recs = get_smart_recommendations(
    "watched too many rom-coms, want something different"
)
print("Avoiding Romance & Comedy:")
for r in recs:
    print(f"  → {r}")

Avoiding Romance & Comedy:
  → ['Ghosts of Mississippi (1996)', 'Man Who Knew Too Much, The (1956)', 'Bicycle Thieves (a.k.a. The Bicycle Thief) (a.k.a. The Bicycle Thieves) (Ladri di biciclette) (1948)', 'S.F.W. (1994)', 'Gettysburg (1993)']
  → Drama


In [20]:
recs = get_smart_recommendations(
    "I liked Vincenzo, suggest similar ones"
)
print("Similar to Vincenzo (Crime):")
for r in recs:
    print(f"  → {r}")

Similar to Vincenzo (Crime):
  → ['Badlands (1973)', 'Mobsters (1991)', 'And Then There Were None (1945)', 'Out of Sight (1998)', 'This World, Then the Fireworks (1997)']
  → Crime


In [21]:
intent = get_user_intent(
    "I watched Vincenzo, suggest different movies"
)
print(intent)

{'mood': None, 'genre_preference': None, 'reference_title': 'Vincenzo', 'similarity_intent': 'different', 'avoid_genres': [], 'vibe': None, 'language': 'Korean'}


In [22]:
# Check genres of the results
for title in ["With a Friend Like Harry... (Harry, un ami qui vous veut du bien) (2000)",
              "Saragossa Manuscript, The (Rekopis znaleziony w Saragossie) (1965)",
              "Center Stage (2000)"]:
    row = df_movies[df_movies['title'] == title]
    if len(row) > 0:
        print(f"{title[:40]}")
        print(f"   Genres: {row['genres'].values[0]}\n")

With a Friend Like Harry... (Harry, un a
   Genres: Drama|Thriller

Saragossa Manuscript, The (Rekopis znale
   Genres: Adventure|Drama|Mystery

Center Stage (2000)
   Genres: Drama|Musical



In [23]:
intent = get_user_intent(
    "watched too many rom-coms, want something different"
)
print("Intent:", intent)

genre = resolve_genre(intent)
print("Genre:", genre)

# Check BEFORE filtering
raw = get_recommendations(genre=genre, n=15, user_id=None)
print(f"\nBefore filter ({len(raw)} movies):")
for r in raw:
    row = df_movies[df_movies['title'] == r]
    if len(row) > 0:
        print(f"  {r[:35]} → {row['genres'].values[0]}")

Intent: {'mood': 'Tired', 'genre_preference': None, 'reference_title': None, 'similarity_intent': 'different', 'avoid_genres': ['Romance', 'Comedy'], 'vibe': None, 'language': None}
Genre: Animation

Before filter (10 movies):
  Hedgehog in the Fog (1975) → Animation
  Dragon Ball Z: Broly - The Legendar → Action|Adventure|Animation
  The Good Dinosaur (2015) → Adventure|Animation|Children|Comedy|Fantasy
  The Amazing Screw-On Head (2006) → Action|Adventure|Animation|Comedy|Sci-Fi
  Storks (2016) → Animation|Children|Comedy
  Oversimplification of Her Beauty, A → Animation|Comedy|Drama|Romance
  Many Adventures of Winnie the Pooh, → Animation|Children|Musical
  A Silent Voice (2016) → Animation|Drama|Romance
  Brother Bear (2003) → Adventure|Animation|Children
  Blood: The Last Vampire (2000) → Action|Animation|Horror


In [24]:


# Test it!
recs = get_smart_recommendations("had a rough day")
print(recs)

(['Hands on a Hard Body (1996)', 'Grown Ups (2010)', 'Crazy in Alabama (1999)', 'Divine Secrets of the Ya-Ya Sisterhood (2002)', 'Father of the Bride Part II (1995)'], 'Comedy')


In [25]:
intent = get_user_intent("feeling tired suggest some feel good movies")
print(intent)

{'mood': 'Tired', 'genre_preference': None, 'reference_title': None, 'similarity_intent': None, 'avoid_genres': [], 'vibe': 'feel-good', 'language': None}


In [26]:
test_vibes = [
    "I want something that makes me smile",
    "need a good revenge story",
    "something touching and emotional",
    "inspire me with a success story",
    "want a tense suspenseful movie",
]

for text in test_vibes:
    intent = get_user_intent(text)
    print(f"INPUT: {text}")
    print(f"VIBE DETECTED: {intent.get('vibe')}")
    print("---")

INPUT: I want something that makes me smile
VIBE DETECTED: feel-good
---
INPUT: need a good revenge story
VIBE DETECTED: revenge
---
INPUT: something touching and emotional
VIBE DETECTED: emotional
---
INPUT: inspire me with a success story
VIBE DETECTED: inspirational
---
INPUT: want a tense suspenseful movie
VIBE DETECTED: suspense
---


In [27]:
intent = get_user_intent("I want something that makes me smile")
print(intent)  # show the FULL dict

{'mood': None, 'genre_preference': None, 'reference_title': None, 'similarity_intent': None, 'avoid_genres': [], 'vibe': 'feel-good', 'language': None}


In [28]:
# Test 2 — Full pipeline
recs = get_smart_recommendations(
    "feeling sad but want a suspenseful movie"
)
print(recs)

(['And Now... Ladies and Gentlemen... (2002)', 'Double, The (2013)', 'Missing (1982)', 'When a Stranger Calls (1979)', 'Films to Keep You Awake: The Christmas Tale (Películas para no dormir: Cuento de navidad) (2005)'], 'Thriller')


In [29]:
# Just mood, NO vibe
recs = get_smart_recommendations("had a rough day")
print(recs)
# Should still give Comedy (mood works!)

(['The Revenant (2015)', 'Crossing Guard, The (1995)', 'Personal Velocity (2002)', 'Man from Snowy River, The (1982)', "It's All Gone Pete Tong (2004)"], 'Drama')


In [30]:
intent = get_user_intent(
    "had a rough day, suggest some feel good movies"
)
print(intent)

{'mood': 'Sad', 'genre_preference': None, 'reference_title': None, 'similarity_intent': None, 'avoid_genres': [], 'vibe': 'feel-good', 'language': None}


In [31]:
intent = get_user_intent("I liked Vincenzo, suggest similar ones")
print(intent)

{'mood': None, 'genre_preference': None, 'reference_title': 'Vincenzo', 'similarity_intent': 'similar', 'avoid_genres': [], 'vibe': None, 'language': 'Korean'}


In [32]:
genre = resolve_genre(intent)
print("Genre chosen:", genre)

Genre chosen: Drama


In [33]:


# Test vibe beats mood
recs = get_smart_recommendations(
    "feeling sad but want suspense"
)
print(recs)

(['Old Boy (2003)', 'Eye for an Eye (1996)', 'Craft, The (1996)', 'Brothers (2009)', 'Airport 1975 (1974)'], 'Thriller')


In [34]:
def generate_explanation(movie, user_context=""):
    """
    Generate a 'Why You'll Like It' reason.
    Accepts EITHER a movie dict (Guest) OR a title string (Personal).
    """
    # Normalize input — figure out title + overview regardless of shape
    if isinstance(movie, dict):
        title = movie.get("title", "")
        overview = movie.get("overview", "")
    else:
        # It's a title string (Personal Mode)
        title = movie
        overview = get_movie_description(title)   # fetch it 🎯

    prompt = f"""
    {user_context}

    Movie: {title}
    Description: {overview}

    Write ONE sentence (max 25 words) explaining why this movie suits how
    they are feeling right now. Connect a specific element of the movie —
    its tone, story, or characters — to their emotional state.
    Begin by acknowledging their state, then give the reason.
    Do NOT summarise the plot. Do NOT include the movie title.
    Return ONLY the sentence.
    """

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content.strip()

In [35]:
# quick test — reuse a movie from discover
movies = discover_movies_by_genre(get_genre_id("Sci-Fi"), n=3)
test_movie = movies[0]

reason = generate_explanation(test_movie, user_context="They're in a Sci-Fi mood.")
print(test_movie["title"], "→", reason)

War Machine → Your sci‑fi mood matches the film’s relentless, otherworldly threat that fuels the same tense, futuristic adrenaline you’re craving.


In [36]:
# Pass a TITLE STRING (Personal Mode style), not a dict
reason = generate_explanation("Interstellar (2014)")
print(reason)

Could you let me know how you’re feeling right now so I can tailor the response to your emotional state?


In [37]:
# Test A — full pipeline, but WITHOUT platform filter
results = guest_recommendations_with_platform(
    genre_name="Drama", user_platforms=None, language="Korean", n=6)
print("No platform filter:", len(results))
for m in results:
    print(" ", m["title"])

No platform filter: 6
  Parasite
  The Handmaiden
  Humint
  20th Century Girl
  The Great Flood
  Even If This Love Disappears Tonight
